# Accounting Standard Changes and the Volatility of Financial Metrics

**Question.** Did the recent transition to forward-looking, current-value accounting frameworks increase the volatility of expected loss of banks and insurers?

* **Banks (IFRS 9 / CECL)** — the Expected Credit Loss (ECL) model ties provisions to macroeconomic forecasts, introducing compounding procyclicality. We proxy bank fundamentals with the **Provision Ratio (PR)** = `loan_loss_provision_ltm / net_loans`.
* **Insurers (IFRS 17 / LDTI)** — a current-value framework lets market interest-rate moves warp operating income. We proxy insurer fundamentals with the **Loss Ratio (LR)** = `insurance_loss_ltm / insurance_premium_ltm`.

**Design.** Two non-overlapping windows on `date_fundamental`:

| Window | Range | `Post` |
|---|---|---|
| Pre-adoption | 2015-01-01 → 2017-12-31 | 0 |
| Post-adoption | 2023-01-01 → 2025-12-31 | 1 |

**Method.** A regression test with firm fixed effects. Because each firm contributes few reports per window, firm-by-firm variance tests lack power; instead we pool firms, measure each observation's absolute deviation from its own *firm-and-period* mean, and regress that deviation on the `Post` dummy. A positive, significant $\beta$ means volatility rose after adoption.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import copy
import polars as pl
import pandas as pd
from plotnine import (
    ggplot, aes, geom_density, facet_wrap,
    labs, theme_bw,
)
from linearmodels.panel import PanelOLS

from src import config as config_mod
from src.data import loaders
from src import universe

cfg = config_mod.load(PROJECT_ROOT / "config.yaml")

# Resolve the data root to an absolute path so the loaders work regardless of
# the kernel's working directory (notebooks/ vs project root).
cfg.raw["data"]["root"] = str((PROJECT_ROOT / cfg["data"]["root"]).resolve())
print("Config loaded. Data root:", cfg["data"]["root"])

## 1. Load data and build the firm panel

We load `fundamental_master_extended` (the bank/insurance line items), `security_master` (GICS classification) and `industry_mapping` (FactSet industry names). The three are joined on `stock_id` to assemble a panel keyed by `(stock_id, date_fundamental)`, carrying exactly the columns `src/universe.py` needs to confirm sector membership.

In [ ]:
ext = loaders.load_table("fundamental_master_extended", cfg)   # bank/insurance metrics
sec = loaders.load_table("security_master", cfg)               # GICS classification (security meta)
ind = loaders.load_table("industry_mapping", cfg)              # FactSet industry names

panel = (
    ext
    .join(sec.select("stock_id", "gics_industry_name"), on="stock_id", how="left")
    .join(ind.select("stock_id", "factset_industry_name"), on="stock_id", how="left")
)
print("Panel columns:", panel.collect_schema().names())

## 2. Filter to the sector universe (banks + insurers)

`universe.sector_set` keeps only securities that are (a) GICS Banks or Insurance, (b) carry a recognised FactSet label, and (c) actually report the sector's defining metric (`net_interest_margin` for banks, `insurance_premium_ltm` for insurers). We then tag each firm with `universe.industry_labels` at **sector** granularity, collapsing the life/P&C split into clean `bank` / `insurance` labels.

In [ ]:
panel = universe.sector_set(panel, cfg)

# Tag bank vs insurance. Override granularity to 'sector' so insurers are not
# split into life / P&C for this analysis.
cfg_sector = copy.deepcopy(cfg)
cfg_sector.raw["universe"]["industry_granularity"] = "sector"
panel = universe.industry_labels(panel, cfg_sector)

## 3. Restrict to the two windows, collapse to quarter-end, build `Post`

`date_fundamental` is month-end point-in-time. The LTM line items refresh only quarterly and are then carried forward unchanged for ~2 months (verified empirically: month-to-month change rates ≈ 1/3 for every metric used here). To remove that redundancy we **collapse each firm's calendar quarter to its last available month-end snapshot**, leaving ~12 genuine observations per firm per window instead of ~36 repeated ones. We then keep only the two windows and flag the post window with `Post = 1`.

In [ ]:
PRE_START,  PRE_END  = pl.date(2015, 1, 1), pl.date(2017, 12, 31)
POST_START, POST_END = pl.date(2023, 1, 1), pl.date(2025, 12, 31)

in_pre  = (pl.col("date_fundamental") >= PRE_START)  & (pl.col("date_fundamental") <= PRE_END)
in_post = (pl.col("date_fundamental") >= POST_START) & (pl.col("date_fundamental") <= POST_END)

panel = (
    panel
    .filter(in_pre | in_post)
    .with_columns(
        Post=pl.when(in_post).then(1).otherwise(0),
        quarter=pl.col("date_fundamental").dt.truncate("1q"),   # calendar-quarter bucket
    )
    # Robustness 2: collapse monthly carried-forward rows to one quarter-end
    # report per firm-quarter (the latest month-end snapshot in each quarter).
    .sort("date_fundamental")
    .unique(subset=["stock_id", "quarter"], keep="last", maintain_order=True)
    .drop("quarter")
    .collect()
)

print(f"Panel rows after quarter-end collapse: {panel.height:,}")
print(panel.group_by(["industry", "Post"]).agg(pl.col("stock_id").n_unique().alias("n_firms"),
                                               pl.len().alias("n_obs")).sort(["industry", "Post"]))

## 4. Construct the normalised target metrics (balanced firms only)

Both ratios are scaled by a balance-sheet / revenue base to remove firm-size effects. We drop rows where the ratio is null or non-finite (e.g. a zero denominator).

**Robustness 1:** after forming the analysis sample we keep only firms observed in *both* the pre- and post-adoption windows. With firm fixed effects the `Post` coefficient is identified purely from within-firm variation across periods, so single-period firms add nothing to $\beta$; restricting to the balanced set makes the pre/post comparison a like-for-like one and keeps the summary statistics on a common set of firms.

In [ ]:
def keep_balanced(df: pl.DataFrame) -> pl.DataFrame:
    """Keep only firms present in BOTH periods (have a Post=0 and a Post=1 row)."""
    return df.filter(
        (pl.col("Post").min().over("stock_id") == 0)
        & (pl.col("Post").max().over("stock_id") == 1)
    )

banks = keep_balanced(
    panel.filter(pl.col("industry") == "bank")
    .with_columns(PR=pl.col("loan_loss_provision_ltm") / pl.col("net_loans"))
    .filter(pl.col("PR").is_finite())   # drops null, +/-inf and NaN
)

insurers = keep_balanced(
    panel.filter(pl.col("industry") == "insurance")
    .with_columns(LR=pl.col("insurance_loss_ltm") / pl.col("insurance_premium_ltm"))
    .filter(pl.col("LR").is_finite())
)

print(f"Bank PR observations:     {banks.height:,}  ({banks['stock_id'].n_unique():,} firms, both periods)")
print(f"Insurer LR observations:  {insurers.height:,}  ({insurers['stock_id'].n_unique():,} firms, both periods)")

## 5. Summary statistics by `Post`

Mean, median and **variance** of each metric, split by period. The variance row is the first informal look at the hypothesis: if the accounting change raised volatility, the post-period variance should be larger.

In [ ]:
def summary_by_post(df: pl.DataFrame, col: str) -> pd.DataFrame:
    out = (
        df.group_by("Post")
        .agg(
            n=pl.len(),
            n_firms=pl.col("stock_id").n_unique(),
            mean=pl.col(col).mean(),
            median=pl.col(col).median(),
            variance=pl.col(col).var(),
            std=pl.col(col).std(),
        )
        .sort("Post")
        .to_pandas()
    )
    out["Period"] = out["Post"].map({0: "Pre (2015-17)", 1: "Post (2023-25)"})
    return out.set_index("Period").drop(columns="Post")

print("=== Bank Provision Ratio (PR) ===")
pr_summary = summary_by_post(banks, "PR")
print("\n=== Insurer Loss Ratio (LR) ===")
lr_summary = summary_by_post(insurers, "LR")
pr_summary

In [ ]:
lr_summary

## 6. Regression test with firm fixed effects

For each metric $Y$:
1. **Demean within firm × period** — subtract each firm's mean over its own observations *within* each period: $\;\tilde Y_{i,t}=Y_{i,t}-\bar Y_{i,\text{Period}}$.
2. **Absolute deviation** — $d_{i,t}=|\tilde Y_{i,t}|$, the magnitude of volatility around the firm's period mean.
3. **Panel OLS with firm fixed effects** — $d_{i,t}=\alpha_i+\beta\,\text{Post}_t+\varepsilon_{i,t}$, standard errors **clustered at the firm-period pair level**.

$\beta$ is the average change in within-firm dispersion from the pre- to the post-adoption window. The firm effects $\alpha_i$ absorb time-invariant firm characteristics; clustering on `stock_id` corrects for the serial correlation induced by the carried-forward monthly observations.

In [ ]:
def levene_fe_regression(df: pl.DataFrame, col: str):
    """Regression-based Levene's test: |Y - firm-period mean| ~ Post + firm FE,
    with SEs clustered by firm-Post pairs. Returns the fitted PanelOLS result."""
    
    work = df.with_columns(
        abs_dev=(pl.col(col) - pl.col(col).mean().over(["stock_id", "Post"])).abs(),
        cluster_group=(pl.col("stock_id").cast(pl.String) + "_" + pl.col("Post").cast(pl.String))
    )
    
    pdf = (
        work
        .select(
            pl.col("stock_id").cast(pl.String),  # plain string index avoids categorical groupby noise
            "date_fundamental",
            "abs_dev",
            "Post",
            "cluster_group"
        )
        .to_pandas()
        .set_index(["stock_id", "date_fundamental"])
    )
    
    model = PanelOLS(pdf["abs_dev"], pdf[["Post"]], entity_effects=True)
    
    return model.fit(
        cov_type="clustered", 
        clusters=pdf[["cluster_group"]] 
    )


def beta_table(result, label: str) -> pd.DataFrame:
    """Compact one-row table: beta on Post, clustered SE, t and p-value."""
    return pd.DataFrame(
        {
            "beta (Post)": [result.params["Post"]],
            "clustered SE": [result.std_errors["Post"]],
            "t-stat": [result.tstats["Post"]],
            "p-value": [result.pvalues["Post"]],
            "n_obs": [int(result.nobs)],
            "n_firms": [int(result.entity_info["total"])],
        },
        index=[label],
    )

### 6a. Banks — Provision Ratio

In [ ]:
bank_res = levene_fe_regression(banks, "PR")
print(bank_res.summary)

### 6b. Insurers — Loss Ratio

In [ ]:
ins_res = levene_fe_regression(insurers, "LR")
print(ins_res.summary)

### 6c. Side-by-side $\beta$ summary

In [ ]:
beta_summary = pd.concat([
    beta_table(bank_res, "Banks: PR (IFRS 9 / CECL)"),
    beta_table(ins_res, "Insurers: LR (IFRS 17 / LDTI)"),
])
beta_summary

## 7. Visual check — distribution of absolute deviations

If volatility rose post-adoption, the absolute-deviation distribution should shift right / fatten in the post window. Axes are clipped to the bulk of the data so a handful of extreme ratios do not dominate the view.

In [ ]:
def abs_dev_frame(df: pl.DataFrame, col: str, sector: str) -> pl.DataFrame:
    return (
        df.with_columns(
            abs_dev=(pl.col(col) - pl.col(col).mean().over(["stock_id", "Post"])).abs()
        )
        .with_columns(
            Period=pl.when(pl.col("Post") == 1).then(pl.lit("Post (2023-25)")).otherwise(pl.lit("Pre (2015-17)")),
            Sector=pl.lit(sector),
        )
        .select("Sector", "Period", "abs_dev")
    )

plot_df = pl.concat([
    abs_dev_frame(banks, "PR", "Banks — Provision Ratio"),
    abs_dev_frame(insurers, "LR", "Insurers — Loss Ratio"),
]).to_pandas()

# Per-sector percentile clip just for display.
clips = plot_df.groupby("Sector")["abs_dev"].quantile(0.95).to_dict()

# Map the percentile clip value to each row
plot_df["clip_val"] = plot_df["Sector"].map(clips)

# Filter out rows where abs_dev exceeds its sector's 99th percentile
plot_df_filtered = plot_df[plot_df["abs_dev"] <= plot_df["clip_val"]]

(
    ggplot(plot_df_filtered, aes(x="abs_dev", fill="Period", color="Period"))
    + geom_density(alpha=0.35)
    + facet_wrap("Sector", scales="free")
    + labs(
        title="Within-firm absolute deviations, pre vs post adoption",
        x="|metric - firm-period mean|", y="density",
    )
    + theme_bw()
)

## 8. Conclusion

**Decision rule.** For each sector, the regression confirms the hypothesis that the accounting change *increased fundamental volatility* when the coefficient $\beta$ on `Post` is **positive and statistically significant** (p < 0.05): post-adoption observations sit, on average, further from their own firm-period mean than pre-adoption ones. A $\beta$ that is negative, or positive but insignificant, fails to confirm the hypothesis. The cell below reads the fitted coefficients and states the verdict for the realised data.

In [ ]:
def verdict(result, sector: str, standard: str, metric: str, alpha: float = 0.05) -> str:
    beta = result.params["Post"]
    p = result.pvalues["Post"]
    sig = p < alpha
    if beta > 0 and sig:
        call = f"CONFIRMS the hypothesis: {standard} is associated with higher {metric} volatility."
    elif beta > 0 and not sig:
        call = f"does NOT confirm: dispersion rose (beta>0) but the effect is not significant at {alpha:.0%}."
    else:
        call = "does NOT confirm: no increase in dispersion (beta<=0) after adoption."
    return (f"{sector} [{standard}] | beta={beta:+.6f}, clustered SE={result.std_errors['Post']:.6f}, "
            f"p={p:.4f}\n    -> {call}")
print(verdict(bank_res, "Banks", "IFRS 9 / CECL", "Provision Ratio"))
print()
print(verdict(ins_res, "Insurers", "IFRS 17 / LDTI", "Loss Ratio"))